# Execution is part of the strategy
Use the course package to generate an Almgren–Chriss inventory path, then reconstruct the continuous-time schedule from its objective parameters.

## 1. Using `finmlsim`
The companion package returns holdings, trades, expected cost, and cost risk.

In [ ]:
import sys, subprocess
try:
    from finmlsim import simulate as sim
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'finmlsim'])
    from finmlsim import simulate as sim
import matplotlib.pyplot as plt

schedule = sim.almgren_chriss(X=1_000_000, T=1, N=20, sigma=.02, eta=2.5e-6, lam=1e-6)
print('first five trades:', schedule['n_trades'][:5].round())
print('expected cost:', round(schedule['E_cost']))
print('cost risk:', round(schedule['sd_cost']))
plt.step(range(21), schedule['x'], where='post'); plt.xlabel('interval'); plt.ylabel('shares remaining'); plt.show();

## 2. Build the schedule from scratch
Use the continuous-time holdings solution and difference adjacent holdings to obtain trades.

In [ ]:
import numpy as np

inventory, horizon, steps = 1_000_000, 1.0, 20
sigma, eta, risk_aversion = .02, 2.5e-6, 1e-6
kappa = np.sqrt(risk_aversion * sigma**2 / eta)
times = np.linspace(0, horizon, steps + 1)
holdings = inventory * np.sinh(kappa * (horizon - times)) / np.sinh(kappa * horizon)
trades = holdings[:-1] - holdings[1:]
print('first five trades:', trades[:5].round())
print('ending inventory:', round(holdings[-1]))

## What to notice
The schedule is the solution to a declared impact-versus-risk problem. Changing risk aversion changes the desired pace even when the forecast is unchanged.